# Teaching an AI to Smell — OpenPOM MPNN Training Pipeline

End-to-end implementation of the **Principal Odor Map (POM)** model from the
Science 2023 paper *"A principal odor map unifies diverse tasks in olfactory perception"*.

**Architecture:** SMILES → Molecular Graph → 5-round MPNN → Attention Pooling → 256-dim POM → 138 Odor Labels

> **Runtime:** Select `Runtime → Change runtime type → T4 GPU` before running.

## Cell 1 — Install Dependencies
Order matters: PyTorch → DGL → RDKit → DeepChem → iterative-stratification

In [ ]:
%%bash
pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html
pip install -q rdkit-pypi
pip install -q --pre deepchem
pip install -q iterative-stratification tqdm seaborn scikit-learn

In [ ]:
import torch
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 2 — Dataset: Load, Featurize, Split
Downloads the curated 4,983-molecule CSV with 138 odor labels from the OpenPOM repo.

In [ ]:
import os, logging, time
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger('aismell')

# ------- Atom & Bond Feature Extraction -------

ATOM_ELEMENTS = ['C','N','O','S','F','Cl','Br','I','P','Si','B','Se','Te','Sn','Ge','Pb','As','Sb','Bi','other']
DEGREES = [0,1,2,3,4,5]
HYBRIDIZATIONS = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                  Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                  Chem.rdchem.HybridizationType.SP3D2]
CHARGES = [-2,-1,0,1,2,3]
NUM_HS = [0,1,2,3,4]
BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

def _one_hot(val, choices):
    enc = [0]*(len(choices)+1)
    try: enc[choices.index(val)] = 1
    except ValueError: enc[-1] = 1
    return enc

def atom_features(atom):
    sym = atom.GetSymbol() if atom.GetSymbol() in ATOM_ELEMENTS else 'other'
    return np.array(
        _one_hot(sym, ATOM_ELEMENTS) + _one_hot(atom.GetDegree(), DEGREES) +
        _one_hot(atom.GetHybridization(), HYBRIDIZATIONS) +
        _one_hot(atom.GetFormalCharge(), CHARGES) +
        _one_hot(atom.GetTotalNumHs(), NUM_HS) +
        [int(atom.GetIsAromatic()), int(atom.IsInRing())], dtype=np.float32)

def bond_features(bond):
    return np.array(
        _one_hot(bond.GetBondType(), BOND_TYPES) +
        [int(bond.GetIsConjugated()), int(bond.IsInRing()),
         int(bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE)], dtype=np.float32)

NODE_DIM = len(atom_features(Chem.MolFromSmiles('C').GetAtomWithIdx(0)))
EDGE_DIM = len(bond_features(Chem.MolFromSmiles('CC').GetBondWithIdx(0)))
print(f'Feature dims: node={NODE_DIM}, edge={EDGE_DIM}')

In [ ]:
# ------- MolGraph class -------

class MolGraph:
    def __init__(self, smiles):
        self.smiles = smiles
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: raise ValueError(f'Cannot parse: {smiles}')
        self.node_features = np.stack([atom_features(a) for a in mol.GetAtoms()])
        src, dst, ef = [], [], []
        for b in mol.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            bf = bond_features(b)
            src += [i, j]; dst += [j, i]; ef += [bf, bf]
        if ef:
            self.edge_index = np.array([src, dst], dtype=np.int64)
            self.edge_features = np.stack(ef)
        else:
            self.edge_index = np.zeros((2,0), dtype=np.int64)
            self.edge_features = np.zeros((0, EDGE_DIM), dtype=np.float32)

# ------- Download & Load Dataset -------

DATA_DIR = Path('data'); DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / 'curated_gs_lf_4983.csv'

if not CSV_PATH.exists():
    import urllib.request
    url = 'https://raw.githubusercontent.com/ARY2260/openpom/main/openpom/data/curated_datasets/curated_gs_lf_4983.csv'
    print(f'Downloading dataset...')
    urllib.request.urlretrieve(url, CSV_PATH)

df = pd.read_csv(CSV_PATH)
SMILES_COL = 'nonStereoSMILES'
LABEL_COLS = [c for c in df.columns if c != SMILES_COL]
print(f'Dataset: {len(df)} molecules, {len(LABEL_COLS)} odor labels')

# Featurize
graphs, labels = [], []
for _, row in df.iterrows():
    try:
        g = MolGraph(row[SMILES_COL])
        graphs.append(g)
        labels.append(row[LABEL_COLS].values.astype(np.float32))
    except: pass
labels = np.stack(labels)
print(f'Featurized: {len(graphs)} molecules')

# Iterative Stratified Split (80/10/10)
indices = np.arange(len(graphs))
msss1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(msss1.split(indices, labels))
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_local, test_local = next(msss2.split(temp_idx, labels[temp_idx]))
val_idx, test_idx = temp_idx[val_local], temp_idx[test_local]

print(f'Split: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}')

# Class weights
pos_counts = labels[train_idx].sum(axis=0)
neg_counts = len(train_idx) - pos_counts
pos_weights = np.clip(neg_counts / np.maximum(pos_counts, 1), 0.1, 100.0).astype(np.float32)
print(f'Pos-weight range: [{pos_weights.min():.1f}, {pos_weights.max():.1f}]')

In [ ]:
# ------- DataLoader Collation -------

import torch
from torch.utils.data import Dataset, DataLoader

class OdorSubset(Dataset):
    def __init__(self, graphs, labels, indices):
        self.graphs = [graphs[i] for i in indices]
        self.labels = labels[indices]
    def __len__(self): return len(self.graphs)
    def __getitem__(self, i): return self.graphs[i], self.labels[i]

def collate_fn(batch):
    gs, ls = zip(*batch)
    nf, ef, src_l, dst_l, bi = [], [], [], [], []
    offset = 0
    for i, g in enumerate(gs):
        nf.append(torch.tensor(g.node_features))
        n_atoms = g.node_features.shape[0]
        if g.edge_features.shape[0] > 0:
            ef.append(torch.tensor(g.edge_features))
            src_l.append(torch.tensor(g.edge_index[0]) + offset)
            dst_l.append(torch.tensor(g.edge_index[1]) + offset)
        bi.append(torch.full((n_atoms,), i, dtype=torch.long))
        offset += n_atoms
    return {
        'node_feats': torch.cat(nf),
        'edge_index': torch.stack([torch.cat(src_l), torch.cat(dst_l)]) if src_l else torch.zeros(2,0,dtype=torch.long),
        'edge_feats': torch.cat(ef) if ef else torch.zeros(0, EDGE_DIM),
        'batch_idx': torch.cat(bi),
        'num_graphs': len(gs),
        'labels': torch.tensor(np.stack(ls), dtype=torch.float32),
    }

BS = 64
train_ds = OdorSubset(graphs, labels, train_idx)
val_ds   = OdorSubset(graphs, labels, val_idx)
test_ds  = OdorSubset(graphs, labels, test_idx)
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BS, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BS, shuffle=False, collate_fn=collate_fn)

b = next(iter(train_loader))
print(f"Sample batch — nodes: {b['node_feats'].shape}, edges: {b['edge_index'].shape}, labels: {b['labels'].shape}")

## Cell 3 — Model Definition: MPNN with Principal Odor Map

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class EdgeNetwork(nn.Module):
    def __init__(self, edge_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(edge_dim, hidden*2), nn.ReLU(), nn.Linear(hidden*2, hidden*hidden))
        self.hidden = hidden
    def forward(self, e):
        return self.net(e).view(-1, self.hidden, self.hidden)

class MPLayer(nn.Module):
    def __init__(self, hidden, edge_dim):
        super().__init__()
        self.edge_nn = EdgeNetwork(edge_dim, hidden)
        self.gru = nn.GRUCell(hidden, hidden)
    def forward(self, h, edge_index, e):
        if e.size(0) == 0: return h
        src, dst = edge_index
        W = self.edge_nn(e)
        msg = torch.bmm(W, h[src].unsqueeze(-1)).squeeze(-1)
        agg = torch.zeros_like(h)
        agg.index_add_(0, dst, msg)
        return self.gru(agg, h)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.transform = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU())
    def forward(self, h, batch_idx, n_graphs):
        g = self.gate(h)
        mx = torch.zeros(n_graphs, 1, device=g.device)
        mx.scatter_reduce_(0, batch_idx.unsqueeze(1), g, reduce='amax')
        g_exp = (g - mx[batch_idx]).exp()
        g_sum = torch.zeros(n_graphs, 1, device=g.device)
        g_sum.scatter_add_(0, batch_idx.unsqueeze(1), g_exp)
        attn = g_exp / g_sum[batch_idx].clamp(min=1e-8)
        weighted = attn * self.transform(h)
        out = torch.zeros(n_graphs, h.size(1), device=h.device)
        out.scatter_add_(0, batch_idx.unsqueeze(1).expand_as(weighted), weighted)
        return out

class MPNNPOMModel(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=128, n_mp=5, pom_dim=256, n_tasks=138, dropout=0.12):
        super().__init__()
        self.node_enc = nn.Linear(node_dim, hidden)
        self.edge_enc = nn.Linear(edge_dim, hidden)
        self.mp_layers = nn.ModuleList([MPLayer(hidden, hidden) for _ in range(n_mp)])
        self.pool = AttentionPool(hidden)
        self.pom = nn.Sequential(nn.Linear(hidden, pom_dim), nn.ReLU(), nn.Dropout(dropout))
        self.head = nn.Linear(pom_dim, n_tasks)
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, batch):
        h = self.node_enc(batch['node_feats'])
        e = self.edge_enc(batch['edge_feats']) if batch['edge_feats'].size(0) > 0 else batch['edge_feats']
        for mp in self.mp_layers:
            h = mp(h, batch['edge_index'], e)
        g = self.pool(h, batch['batch_idx'], batch['num_graphs'])
        return self.head(self.pom(g))

    def extract_pom(self, batch):
        h = self.node_enc(batch['node_feats'])
        e = self.edge_enc(batch['edge_feats']) if batch['edge_feats'].size(0) > 0 else batch['edge_feats']
        for mp in self.mp_layers:
            h = mp(h, batch['edge_index'], e)
        g = self.pool(h, batch['batch_idx'], batch['num_graphs'])
        return self.pom(g)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MPNNPOMModel(NODE_DIM, EDGE_DIM).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params:,} parameters on {device}')

## Cell 4 — Training Loop

In [ ]:
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

def macro_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        col = y_true[:, i]
        if col.sum() == 0 or col.sum() == len(col): continue
        try: aucs.append(roc_auc_score(col, y_pred[:, i]))
        except: pass
    return np.mean(aucs) if aucs else 0.5

def to_dev(batch, dev):
    return {k: v.to(dev) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

# --- Hyperparameters ---
EPOCHS = 5        # Set to 150 for full benchmark
LR = 1e-3
GRAD_CLIP = 5.0

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, device=device))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

best_auc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_loss = 0; n_batch = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
        batch = to_dev(batch, device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                loss = criterion(model(batch), batch['labels'])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(batch), batch['labels'])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        epoch_loss += loss.item(); n_batch += 1
    scheduler.step()
    avg_train_loss = epoch_loss / max(n_batch, 1)

    # --- Validate ---
    model.eval()
    all_y, all_p = [], []
    v_loss = 0; v_n = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = to_dev(batch, device)
            logits = model(batch)
            v_loss += criterion(logits, batch['labels']).item(); v_n += 1
            all_y.append(batch['labels'].cpu().numpy())
            all_p.append(torch.sigmoid(logits).cpu().numpy())
    all_y = np.concatenate(all_y); all_p = np.concatenate(all_p)
    val_auc = macro_auc(all_y, all_p)
    avg_val_loss = v_loss / max(v_n, 1)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_auc'].append(val_auc)

    marker = ''
    if val_auc > best_auc:
        best_auc = val_auc; marker = ' [BEST]'
        torch.save(model.state_dict(), 'best_model.pt')

    print(f'Epoch {epoch:3d}/{EPOCHS}  train_loss={avg_train_loss:.4f}  '
          f'val_loss={avg_val_loss:.4f}  val_AUC={val_auc:.4f}{marker}')

print(f'\nBest validation AUC: {best_auc:.4f}')

## Cell 5 — Test Evaluation

In [ ]:
model.load_state_dict(torch.load('best_model.pt', weights_only=True))
model.eval()
all_y, all_p = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = to_dev(batch, device)
        all_y.append(batch['labels'].cpu().numpy())
        all_p.append(torch.sigmoid(model(batch)).cpu().numpy())
all_y = np.concatenate(all_y); all_p = np.concatenate(all_p)
test_auc = macro_auc(all_y, all_p)
print(f'Test Macro ROC-AUC: {test_auc:.4f}')

## Cell 6 — Predict: What Does This Molecule Smell Like?

In [ ]:
def predict_smell(smiles_str, top_k=10):
    """Predict top-k odor descriptors for a SMILES string."""
    try:
        g = MolGraph(smiles_str)
    except ValueError as e:
        print(f'Error: {e}')
        return

    dummy = np.zeros(len(LABEL_COLS), dtype=np.float32)
    batch = collate_fn([(g, dummy)])
    batch = to_dev(batch, device)

    model.eval()
    with torch.no_grad():
        logits = model(batch)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        pom_vec = model.extract_pom(batch).cpu().numpy()[0]

    top_idx = np.argsort(probs)[::-1][:top_k]
    print(f'\n{"="*55}')
    print(f'  SMILES: {smiles_str}')
    print(f'{"="*55}')
    print(f'  {"Rank":<5} {"Descriptor":<22} {"Prob":>8}')
    print(f'  {"─"*5} {"─"*22} {"─"*8}')
    for r, i in enumerate(top_idx, 1):
        bar = '█' * int(probs[i]*20)
        print(f'  {r:<5} {LABEL_COLS[i]:<22} {probs[i]:>8.4f} {bar}')
    print(f'\n  POM vector norm: {np.linalg.norm(pom_vec):.4f}')
    return probs, pom_vec

# --- Try some molecules ---
predict_smell('CCO')              # Ethanol
predict_smell('c1ccccc1')         # Benzene
predict_smell('CC(=O)O')          # Acetic acid (vinegar)
predict_smell('C=CC(=O)OCC')      # Ethyl acrylate

## Cell 7 — Visualize the Principal Odor Map (POM)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Extract POM vectors for all test molecules
pom_vecs, test_labels = [], []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = to_dev(batch, device)
        pv = model.extract_pom(batch).cpu().numpy()
        pom_vecs.append(pv)
        test_labels.append(batch['labels'].cpu().numpy())
pom_vecs = np.concatenate(pom_vecs)
test_labels = np.concatenate(test_labels)

# PCA to 2D
pca = PCA(n_components=2)
pom_2d = pca.fit_transform(pom_vecs)

# Color by a few descriptors
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, desc in zip(axes, ['fruity', 'smoky', 'sweet']):
    if desc in LABEL_COLS:
        idx = LABEL_COLS.index(desc)
        colors = test_labels[:, idx]
        sc = ax.scatter(pom_2d[:,0], pom_2d[:,1], c=colors, cmap='RdYlGn', s=8, alpha=0.7)
        ax.set_title(f'POM colored by "{desc}"')
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        plt.colorbar(sc, ax=ax)
plt.suptitle('Principal Odor Map — 2D PCA Projection', fontsize=14)
plt.tight_layout()
plt.savefig('pom_visualization.png', dpi=150)
plt.show()
print(f'PCA explained variance: {pca.explained_variance_ratio_}')

## Cell 8 — Training Loss & AUC Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['val_auc'], label='Val Macro AUC', color='green')
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro ROC-AUC')
ax2.set_title('Validation AUC'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

---
## Next Steps

1. **Full benchmark:** Set `EPOCHS = 150` in Cell 4 and re-run
2. **10-model ensemble:** Re-run the training loop with 10 different seeds (43–52), save each `best_model.pt`, and average their sigmoid outputs at inference
3. **Target:** ~0.89 Macro ROC-AUC with the ensemble
4. **Hardware nose:** Connect a Bosch BME688 sensor via I2C to a Raspberry Pi for real-world gas sensing